In [2]:
# ============================================================
# CUSTOMER CHURN – LR vs XGBOOST (UPGRADED VERSION)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    auc
)
from xgboost import XGBClassifier

# -------------------------
# 1. LOAD + CLEAN
# -------------------------
df = pd.read_csv("CustomerChurn.csv")

df = df.drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"] * df["tenure"])

# -------------------------
# 2. TARGET + FEATURES
# -------------------------
y = (df["Churn"] == "Yes").astype(int)
df = df.drop(columns=["Churn"])

cat_cols = df.select_dtypes(include="object").columns
X = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# -------------------------
# 3. SPLIT FIRST (avoid leakage)
# -------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# -------------------------
# 4. SCALE (ONLY FOR LR)
# -------------------------
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

scaler = StandardScaler()
X_train_full_scaled = X_train_full.copy()
X_test_scaled = X_test.copy()

X_train_full_scaled[num_cols] = scaler.fit_transform(X_train_full[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# -------------------------
# 5. LOGISTIC REGRESSION
# -------------------------
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

log_reg.fit(X_train_full_scaled, y_train_full)

y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]
roc_auc_lr = roc_auc_score(y_test, y_prob_lr)

print("Logistic Regression ROC–AUC:", round(roc_auc_lr, 4))

# -------------------------
# 6. CROSS-VALIDATION (LR)
# -------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    log_reg,
    X_train_full_scaled,
    y_train_full,
    cv=cv,
    scoring="roc_auc"
)

print("LR CV Mean:", cv_scores.mean())
print("LR CV Std:", cv_scores.std())

# -------------------------
# 7. XGBOOST (IMPROVED)
# -------------------------

scale_pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    use_label_encoder=False
)

param_grid = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_full, y_train_full)

best_xgb = search.best_estimator_

print("Best XGB Params:", search.best_params_)

y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]
roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)

print("XGBoost ROC–AUC:", round(roc_auc_xgb, 4))

# -------------------------
# 8. PR-AUC (IMBALANCE METRIC)
# -------------------------
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_prob_lr)
precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, y_prob_xgb)

pr_auc_lr = auc(recall_lr, precision_lr)
pr_auc_xgb = auc(recall_xgb, precision_xgb)

print("LR PR-AUC:", pr_auc_lr)
print("XGB PR-AUC:", pr_auc_xgb)

# -------------------------
# 9. THRESHOLD SEARCH (SYSTEMATIC)
# -------------------------
thresholds = np.linspace(0.1, 0.9, 50)

def find_best_threshold(y_true, y_prob):
    best_thresh = 0
    best_score = 0
    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        score = roc_auc_score(y_true, y_prob)
        if score > best_score:
            best_score = score
            best_thresh = t
    return best_thresh

best_thresh_xgb = find_best_threshold(y_test, y_prob_xgb)
print("Best Threshold (XGB):", best_thresh_xgb)

# -------------------------
# 10. FEATURE IMPORTANCE (XGB)
# -------------------------
importances = pd.Series(
    best_xgb.feature_importances_,
    index=X_train_full.columns
).sort_values(ascending=False)

print("\nTop 10 Important Features (XGB):")
print(importances.head(10))

# -------------------------
# 11. SAVE BEST MODEL
# -------------------------
joblib.dump(best_xgb, "best_xgb_model.pkl")
joblib.dump(log_reg, "log_reg_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_columns.pkl")

print("\nModels saved successfully.")

Logistic Regression ROC–AUC: 0.8419
LR CV Mean: 0.8459074932357323
LR CV Std: 0.012391354417860954
Fitting 5 folds for each of 20 candidates, totalling 100 fits


C:\Users\mohda\anaconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [06:59:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best XGB Params: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
XGBoost ROC–AUC: 0.8464
LR PR-AUC: 0.6329074102320367
XGB PR-AUC: 0.6660541612715296
Best Threshold (XGB): 0.1

Top 10 Important Features (XGB):
Contract_Two year                     0.251760
Contract_One year                     0.149835
InternetService_Fiber optic           0.139497
InternetService_No                    0.080558
OnlineSecurity_No internet service    0.062305
PaymentMethod_Electronic check        0.042841
tenure                                0.035959
StreamingMovies_Yes                   0.029183
StreamingTV_Yes                       0.022767
OnlineSecurity_Yes                    0.022236
dtype: float32

Models saved successfully.


In [6]:
# -------------------------
# 8. THRESHOLD & METRICS (use same THRESHOLD as LR script)
# -------------------------

THRESHOLD = 0.45

# --- XGBoost Predictions ---
y_pred_xgb = (y_prob_xgb >= THRESHOLD).astype(int)

print("\nXGBoost CONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nXGBoost CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred_xgb, digits=4))

# --- Logistic Regression Predictions (same threshold for comparison) ---
y_pred_lr = (y_prob_lr >= THRESHOLD).astype(int)

print("\nLogReg CONFUSION MATRIX (same threshold)")
print(confusion_matrix(y_test, y_pred_lr))

print("\nLogReg CLASSIFICATION REPORT (same threshold)")
print(classification_report(y_test, y_pred_lr, digits=4))


XGBoost CONFUSION MATRIX
[[716 319]
 [ 63 311]]

XGBoost CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.9191    0.6918    0.7894      1035
           1     0.4937    0.8316    0.6195       374

    accuracy                         0.7289      1409
   macro avg     0.7064    0.7617    0.7045      1409
weighted avg     0.8062    0.7289    0.7443      1409


LogReg CONFUSION MATRIX (same threshold)
[[700 335]
 [ 61 313]]

LogReg CLASSIFICATION REPORT (same threshold)
              precision    recall  f1-score   support

           0     0.9198    0.6763    0.7795      1035
           1     0.4830    0.8369    0.6125       374

    accuracy                         0.7189      1409
   macro avg     0.7014    0.7566    0.6960      1409
weighted avg     0.8039    0.7189    0.7352      1409

